# fase_2 - script_afrida Migration

This notebook handles migration of database from old DB to new DB for fase 2.

**Purpose**: Benerin database lama ke database baru untuk bagian [NAMA TABEL]

In [1]:
import sys
import os
import mysql.connector
import pandas as pd
sys.path.append(os.path.abspath('..'))
from config import get_db_config
import warnings
warnings.filterwarnings('ignore')

## 1. Connect ke Database

In [2]:
# Connect ke database config
config = get_db_config()
# Ambil host dari salah satu config (misal db_old)
print(f'Database config loaded: {config["db_old"]["host"]}')

# Connect ke DB Lama
db_old = mysql.connector.connect(**config['db_old'])
cursor_old = db_old.cursor(dictionary=True)
print(f'Connected to old database: {config["db_old"]["database"]}')

# Connect ke DB Baru
db_new = mysql.connector.connect(**config['db_new'])
cursor_new = db_new.cursor(dictionary=True)
print(f'Connected to new database: {config["db_new"]["database"]}')


Database config loaded: 100.75.213.18
Connected to old database: dataleap_v5_example
Connected to new database: dataleap_v5_migration


## 2. Ambil Data dari DB Lama

In [3]:
def cek_kualitas(df, nama_tabel, pk_col):
    print(f"\n{'='*50}")
    print(f"  CEK DATA: {nama_tabel}")
    print(f"{'='*50}")
    print("Missing values:")
    print(df.isnull().sum())
    print(f"\nDuplikat {pk_col}: {df[pk_col].duplicated().sum()} baris")
    print("\nSample data (5 baris pertama):")
    
    display(df.head())

In [4]:
df_periode_lama = pd.read_sql('SELECT * FROM periode', db_old)
cek_kualitas(df_periode_lama, 'periode (lama)', 'idperiode')

print(f"Jumlah data awal: {len(df_periode_lama)}")


  CEK DATA: periode (lama)
Missing values:
idperiode        0
nama_term        0
tanggal          0
bulan_awal       0
tahun_awal       0
idpendkursus     0
jml_sesi         0
tahun_ajar       0
mitra           93
dtype: int64

Duplikat idperiode: 0 baris

Sample data (5 baris pertama):


,idperiode,nama_term,tanggal,bulan_awal,tahun_awal,idpendkursus,jml_sesi,tahun_ajar,mitra
0,P00006,General English Term I July-October 2023,4.0,Juli,2023.0,K00001,30.0,2023/2024,None
1,P00008,General English Term II Oct '23 - Feb '24,25.0,Oktober,2023.0,K00001,30.0,2023/2024,None
2,P00009,General English Term III Feb-Jun 2024,21.0,Februari,2024.0,K00001,30.0,2023/2024,None
3,P00010,Coding Semester I-2023/2024,1.0,Agustus,2023.0,K00002,18.0,2023/2024,None
4,P00011,Coding Semester II-2023/2024,23.0,Januari,2024.0,K00002,18.0,2023/2024,None


Jumlah data awal: 93


In [6]:
# =================================================
# CEK ANOMALI - periode (data mentah dari db_old)
# =================================================
# Pastikan df_periode_lama sudah ada, atau tarik ulang:
if 'df_periode_lama' not in globals():
    df_periode_lama = pd.read_sql('SELECT * FROM periode', engine_old)

print("Jumlah data awal:", len(df_periode_lama))
print("Kolom:", df_periode_lama.columns.tolist())
display(df_periode_lama.head(10))

print("\n=== 1. Missing Values ===")
print(df_periode_lama.isnull().sum())

print("\n=== 2. Duplikat idperiode ===")
dup = df_periode_lama[df_periode_lama['idperiode'].duplicated(keep=False)]
if len(dup) > 0:
    print(f"⚠️ Ada {len(dup)} baris dengan idperiode duplikat:")
    print(dup[['idperiode', 'nama_term']].head(10))
else:
    print("✅ Tidak ada duplikat idperiode.")

print("\n=== 3. Format idperiode (harusnya P diikuti 5 digit) ===")
import re
pola_p = re.compile(r'^P\d{5}$')
mask_p = df_periode_lama['idperiode'].apply(lambda x: bool(pola_p.match(str(x))))
if not mask_p.all():
    print("⚠️ idperiode menyimpang:")
    print(df_periode_lama[~mask_p][['idperiode']].head())
else:
    print("✅ Semua idperiode sesuai format.")

print("\n=== 4. Cek kolom tanggal (hari) ===")
print("Tipe:", df_periode_lama['tanggal'].dtype)
print("Min:", df_periode_lama['tanggal'].min(), "Max:", df_periode_lama['tanggal'].max())
# Cek apakah semua masuk akal (1-31)
invalid_tgl = df_periode_lama[~df_periode_lama['tanggal'].between(1,31)]
if len(invalid_tgl) > 0:
    print(f"⚠️ {len(invalid_tgl)} tanggal di luar rentang 1-31:")
    print(invalid_tgl[['idperiode', 'tanggal']].head())
else:
    print("✅ Semua tanggal dalam rentang 1-31.")

print("\n=== 5. Cek kolom bulan_awal (nama bulan Indonesia) ===")
bulan_valid = {'Januari','Februari','Maret','April','Mei','Juni',
               'Juli','Agustus','September','Oktober','November','Desember'}
bulan_unik = set(df_periode_lama['bulan_awal'].dropna().unique())
bulan_aneh = bulan_unik - bulan_valid
if bulan_aneh:
    print(f"⚠️ Nama bulan tidak dikenali: {bulan_aneh}")
else:
    print("✅ Semua bulan_awal valid.")

print("\n=== 6. Cek kolom tahun_awal ===")
print("Tipe:", df_periode_lama['tahun_awal'].dtype)
print("Min:", df_periode_lama['tahun_awal'].min(), "Max:", df_periode_lama['tahun_awal'].max())
# tahun seharusnya integer 4 digit, pastikan tidak ada 0 atau aneh
invalid_tahun = df_periode_lama[df_periode_lama['tahun_awal'] < 2000]
if len(invalid_tahun) > 0:
    print("⚠️ Tahun di bawah 2000:")
    print(invalid_tahun[['idperiode', 'tahun_awal']].head())

print("\n=== 7. Cek kolom idpendkursus (FK ke kursus) ===")
# Format biasanya K diikuti 5 digit
pola_k = re.compile(r'^K\d{5}$')
mask_k = df_periode_lama['idpendkursus'].apply(lambda x: bool(pola_k.match(str(x))))
if not mask_k.all():
    print("⚠️ idpendkursus menyimpang:")
    print(df_periode_lama[~mask_k][['idpendkursus']].head())
else:
    print("✅ Format idpendkursus sesuai.")
# Validasi ke tabel kursus di db_old
try:
    kursus_old = pd.read_sql('SELECT idpendkursus FROM kursus', db_old)
    set_kursus = set(kursus_old['idpendkursus'].unique())
    missing_fk = df_periode_lama[~df_periode_lama['idpendkursus'].isin(set_kursus)]
    if len(missing_fk) > 0:
        print(f"⚠️ {len(missing_fk)} idpendkursus tidak ada di tabel kursus:")
        print(missing_fk[['idperiode', 'idpendkursus']].head())
    else:
        print("✅ Semua idpendkursus valid (ada di tabel kursus).")
except Exception as e:
    print(f"ℹ️ Tidak bisa validasi FK: {e}")

print("\n=== 8. jml_sesi ===")
print("Tipe:", df_periode_lama['jml_sesi'].dtype)
print("Min:", df_periode_lama['jml_sesi'].min(), "Max:", df_periode_lama['jml_sesi'].max())
# harusnya integer > 0, disini float
invalid_sesi = df_periode_lama[df_periode_lama['jml_sesi'] <= 0]
if len(invalid_sesi) > 0:
    print(f"⚠️ {len(invalid_sesi)} baris dengan jml_sesi <= 0")
else:
    print("✅ jml_sesi > 0.")

print("\n=== 9. tahun_ajar ===")
print("Unique values:", df_periode_lama['tahun_ajar'].unique())
# format biasanya 'YYYY/YYYY'
pola_ajar = re.compile(r'^\d{4}/\d{4}$')
mask_ajar = df_periode_lama['tahun_ajar'].apply(lambda x: bool(pola_ajar.match(str(x))))
if not mask_ajar.all():
    print("⚠️ Format tahun_ajar tidak sesuai:")
    print(df_periode_lama[~mask_ajar][['idperiode', 'tahun_ajar']].head())
else:
    print("✅ Format tahun_ajar sesuai.")

print("\n=== 10. mitra ===")
print("Unique values:", df_periode_lama['mitra'].unique())
# harusnya semua None/NaN, karena nanti dihapus
if df_periode_lama['mitra'].notna().any():
    print("⚠️ Ada nilai di kolom mitra, padahal harusnya kosong semua.")
else:
    print("✅ Kolom mitra kosong semua, aman dihapus.")

Jumlah data awal: 93
Kolom: ['idperiode', 'nama_term', 'tanggal', 'bulan_awal', 'tahun_awal', 'idpendkursus', 'jml_sesi', 'tahun_ajar', 'mitra']


,idperiode,nama_term,tanggal,bulan_awal,tahun_awal,idpendkursus,jml_sesi,tahun_ajar,mitra
0,P00006,General English Term I July-October 2023,4.0,Juli,2023.0,K00001,30.0,2023/2024,None
1,P00008,General English Term II Oct '23 - Feb '24,25.0,Oktober,2023.0,K00001,30.0,2023/2024,None
2,P00009,General English Term III Feb-Jun 2024,21.0,Februari,2024.0,K00001,30.0,2023/2024,None
3,P00010,Coding Semester I-2023/2024,1.0,Agustus,2023.0,K00002,18.0,2023/2024,None
4,P00011,Coding Semester II-2023/2024,23.0,Januari,2024.0,K00002,18.0,2023/2024,None
5,P00012,CC Kids Term Jul-Agt 2023/2024,8.0,Juli,2023.0,K00004,8.0,2023/2024,None
6,P00015,LLC TUE Term 2,4.0,Juli,2023.0,K00003,15.0,2023/2024,None
7,P00016,LLC WED Term 2,5.0,Juli,2023.0,K00003,15.0,2023/2024,None
8,P00017,LLC Term 3 2023/2024,1.0,November,2023.0,K00003,15.0,2023/2024,None
9,P00018,LLC NEW 2024/2025,8.0,Juli,2024.0,K00003,43.0,2024/2025,None



=== 1. Missing Values ===
idperiode        0
nama_term        0
tanggal          0
bulan_awal       0
tahun_awal       0
idpendkursus     0
jml_sesi         0
tahun_ajar       0
mitra           93
dtype: int64

=== 2. Duplikat idperiode ===
✅ Tidak ada duplikat idperiode.

=== 3. Format idperiode (harusnya P diikuti 5 digit) ===
✅ Semua idperiode sesuai format.

=== 4. Cek kolom tanggal (hari) ===
Tipe: float64
Min: 1.0 Max: 26.0
✅ Semua tanggal dalam rentang 1-31.

=== 5. Cek kolom bulan_awal (nama bulan Indonesia) ===
✅ Semua bulan_awal valid.

=== 6. Cek kolom tahun_awal ===
Tipe: float64
Min: 2023.0 Max: 2026.0

=== 7. Cek kolom idpendkursus (FK ke kursus) ===
✅ Format idpendkursus sesuai.
ℹ️ Tidak bisa validasi FK: 1054 (42S22): Unknown column 'idpendkursus' in 'field list'

=== 8. jml_sesi ===
Tipe: float64
Min: 4.0 Max: 43.0
✅ jml_sesi > 0.

=== 9. tahun_ajar ===
Unique values: <StringArray>
['2023/2024', '2024/2025', '2025/2026', '2026', '2026/2027', '20252026']
Length: 6, dty

In [7]:
# =================================================
# PROSES TABEL: periode
# =================================================

# 2. Mapping nama bulan Indonesia → angka
bulan_map = {
    'Januari': 1, 'Februari': 2, 'Maret': 3,
    'April': 4, 'Mei': 5, 'Juni': 6,
    'Juli': 7, 'Agustus': 8, 'September': 9,
    'Oktober': 10, 'November': 11, 'Desember': 12
}

# 3. Konstruksi tanggal_mulai dari tanggal, bulan_awal, tahun_awal
def buat_tanggal(row):
    try:
        hari = int(row['tanggal'])
        bulan = bulan_map[row['bulan_awal']]
        tahun = int(row['tahun_awal'])
        # Gabung jadi string 'YYYY-MM-DD' lalu parse
        date_str = f"{tahun}-{bulan:02d}-{hari:02d}"
        return pd.to_datetime(date_str).date()
    except:
        # Jika ada yang kosong atau error, isi dengan NaT (kita bisa investigasi lebih lanjut)
        return pd.NaT

df_periode_lama['tanggal_mulai'] = df_periode_lama.apply(buat_tanggal, axis=1)

# 4. Hapus kolom lama yang tidak diperlukan
kolom_dibuang = ['tanggal', 'bulan_awal', 'tahun_awal', 'mitra']
df_periode = df_periode_lama.drop(columns=kolom_dibuang, errors='ignore')

# 5. Rename kolom-kolom yang tersisa
mapping_periode = {
    'idperiode': 'id_periode',
    'nama_term': 'nama_periode',
    'idpendkursus': 'id_kursus',
    'jml_sesi': 'jumlah_sesi',
    'tahun_ajar': 'tahun_ajar'        # biarkan sama dulu
}
df_periode = df_periode.rename(columns=mapping_periode)

# 6. Tambah kolom baru
df_periode['status'] = 1
df_periode['is_active'] = 1

# 7. Pastikan tipe data
df_periode['id_periode'] = df_periode['id_periode'].astype(object)  # kode seperti 'P00006', jadi str
df_periode['id_kursus'] = df_periode['id_kursus'].astype(object)    # jika di DB baru juga varchar, sesuaikan
df_periode['jumlah_sesi'] = df_periode['jumlah_sesi'].astype(int)
df_periode['tahun_ajar'] = df_periode['tahun_ajar'].astype(object)
# tanggal_mulai sudah bertipe date

In [8]:
# 8. Cek kualitas
cek_kualitas(df_periode, 'periode', 'id_periode')

# Cek tanggal yang gagal di-parse (NaT)
gagal = df_periode[df_periode['tanggal_mulai'].isna()]
if len(gagal) > 0:
    print(f"\n⚠️  {len(gagal)} baris gagal membuat tanggal_mulai:")
    print(gagal[['id_periode', 'nama_periode']])

# Cek FK ke tabel kursus di db_old (karena db_new belum ada isinya)
try:
    # Asumsi di db_old tabel kursus punya primary key 'idpendkursus'
    kursus_old = pd.read_sql('SELECT idpendkursus FROM pendidikankursus', db_old)
    id_kursus_set = set(kursus_old['idpendkursus'].unique())
    missing_fk = df_periode[~df_periode['id_kursus'].isin(id_kursus_set)]
    if len(missing_fk) > 0:
        print(f"\n⚠️  Ditemukan {len(missing_fk)} id_kursus di periode yang TIDAK ada di tabel kursus (db_old):")
        print(missing_fk[['id_periode', 'id_kursus']].head())
    else:
        print("✅ Semua id_kursus di periode valid (ditemukan di db_old).")
except Exception as e:
    print(f"ℹ️  Gagal validasi FK ke kursus via db_old: {e}")


  CEK DATA: periode
Missing values:
id_periode       0
nama_periode     0
id_kursus        0
jumlah_sesi      0
tahun_ajar       0
tanggal_mulai    0
status           0
is_active        0
dtype: int64

Duplikat id_periode: 0 baris

Sample data (5 baris pertama):


,id_periode,nama_periode,id_kursus,jumlah_sesi,tahun_ajar,tanggal_mulai,status,is_active
0,P00006,General English Term I July-October 2023,K00001,30,2023/2024,2023-07-04,1,1
1,P00008,General English Term II Oct '23 - Feb '24,K00001,30,2023/2024,2023-10-25,1,1
2,P00009,General English Term III Feb-Jun 2024,K00001,30,2023/2024,2024-02-21,1,1
3,P00010,Coding Semester I-2023/2024,K00002,18,2023/2024,2023-08-01,1,1
4,P00011,Coding Semester II-2023/2024,K00002,18,2023/2024,2024-01-23,1,1


✅ Semua id_kursus di periode valid (ditemukan di db_old).


In [13]:
# Hapus baris yang id_kursus-nya K00017
sebelum = len(df_periode)
df_periode = df_periode[df_periode['id_kursus'] != 'K00017']
print(f"Baris dihapus: {sebelum - len(df_periode)}")

Baris dihapus: 2


In [15]:
# Cek lagi missing FK
kursus_valid = pd.read_sql("SELECT id_kursus FROM kursus", db_new)['id_kursus'].tolist()
missing = df_periode[~df_periode['id_kursus'].isin(kursus_valid)]
if len(missing) == 0:
    print("✅ Semua id_kursus sekarang valid.")
else:
    print(f"⚠️ Masih ada {len(missing)} yang tidak valid:")
    display(missing[['id_periode', 'id_kursus']])

✅ Semua id_kursus sekarang valid.


In [16]:
# Pastikan id_kursus sudah string dan tidak ada spasi
df_periode['id_kursus'] = df_periode['id_kursus'].astype(str).str.strip()

# Ambil daftar id_kursus yang ada di DB baru (pakai engine_new)
kursus_valid = pd.read_sql("SELECT id_kursus FROM kursus", db_new)['id_kursus'].tolist()
print("Contoh id_kursus di DB baru:", kursus_valid[:5])

# Cek id_kursus di periode yang TIDAK ada di kursus
missing = df_periode[~df_periode['id_kursus'].isin(kursus_valid)]
if len(missing) > 0:
    print(f"⚠️ Ditemukan {len(missing)} id_kursus yang tidak valid:")
    display(missing[['id_periode', 'id_kursus']].head())
else:
    print("✅ Semua id_kursus di periode cocok dengan tabel kursus.")

Contoh id_kursus di DB baru: ['K00001', 'K00002', 'K00003', 'K00004', 'K00005']
✅ Semua id_kursus di periode cocok dengan tabel kursus.


In [18]:
# =================================================
# PROSES TABEL: parameter_nilai
# =================================================

# 1. Tarik data
df_param_lama = pd.read_sql('SELECT * FROM parameter_nilai', db_old)
print(f"Jumlah data awal: {len(df_param_lama)}")
print("Kolom:", df_param_lama.columns.tolist())
display(df_param_lama.head(10))

# 2. Cek tipe data & missing
print("\nInfo:")
print(df_param_lama.info())
print("\nMissing values:")
print(df_param_lama.isnull().sum())
print("\nNilai unik isnumber:")
print(df_param_lama['isnumber'].unique())
print("\nNilai unik idlevel:")
print(df_param_lama['idlevel'].unique())

Jumlah data awal: 1187
Kolom: ['idp_nilai', 'idlevel', 'parameter', 'isnumber']


,idp_nilai,idlevel,parameter,isnumber
0,P00745,L00022,Class participation,0.0
1,P00746,L00022,Oral,0.0
2,P00747,L00022,Listening,0.0
3,P00748,L00022,Writing,0.0
4,P00749,L00022,Writing-1,1.0
5,P00750,L00022,Grammar Reading-1,1.0
6,P00751,L00022,Presentation-1,1.0
7,P00752,L00022,Listening-1,1.0
8,P00753,L00022,Writing-2,1.0
9,P00754,L00022,Grammar Reading-2,1.0



Info:
<class 'pandas.DataFrame'>
RangeIndex: 1187 entries, 0 to 1186
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   idp_nilai  1187 non-null   str    
 1   idlevel    1187 non-null   str    
 2   parameter  1187 non-null   str    
 3   isnumber   1187 non-null   float64
dtypes: float64(1), str(3)
memory usage: 37.2 KB
None

Missing values:
idp_nilai    0
idlevel      0
parameter    0
isnumber     0
dtype: int64

Nilai unik isnumber:
[0. 1.]

Nilai unik idlevel:
<StringArray>
['L00022', 'L00011', 'L00001', 'L00045', 'L00047', 'L00002', 'L00003',
 'L00004', 'L00005', 'L00006',
 ...
 'L00168', 'L00169', 'L00170', 'L00171', 'L00172', 'L00173', 'L00174',
 'L00158', 'L00184', 'L00185']
Length: 142, dtype: str


In [19]:

# 2. Mapping kolom
mapping_param = {
    'idp_nilai': 'id_parameter_nilai',
    'idlevel': 'id_level',
    'parameter': 'nama_parameter',
    'isnumber': 'status_parameter'
}
df_param = df_param_lama.rename(columns=mapping_param)

# 3. Konversi tipe data
df_param['id_parameter_nilai'] = df_param['id_parameter_nilai'].astype(str)   # tetap VARCHAR
df_param['id_level'] = df_param['id_level'].astype(str)                       # VARCHAR juga
df_param['nama_parameter'] = df_param['nama_parameter'].astype(str)
# status_parameter: float -> int (sudah 0.0 / 1.0, aman dikonversi)
df_param['status_parameter'] = df_param['status_parameter'].astype(int)      # jadi tinyint

# 4. Cek kualitas
cek_kualitas(df_param, 'parameter_nilai', 'id_parameter_nilai')

# 5. Validasi Foreign Key `id_level` ke tabel `level` di db_old
try:
    level_old = pd.read_sql('SELECT idlevel FROM level', db_old)  # sesuaikan nama kolom PK level
    id_level_set = set(level_old['idlevel'].unique())
    missing_fk = df_param[~df_param['id_level'].isin(id_level_set)]
    if len(missing_fk) > 0:
        print(f"\n⚠️  Ditemukan {len(missing_fk)} id_level di parameter_nilai yang TIDAK ada di tabel level (db_old):")
        print(missing_fk[['id_parameter_nilai', 'id_level']].head())
    else:
        print("✅ Semua id_level di parameter_nilai valid (ditemukan di db_old).")
except Exception as e:
    print(f"ℹ️  Gagal validasi FK ke level via db_old: {e}")
    print("   Lanjut tanpa validasi FK. Pastikan tabel level sudah ada nanti.")



  CEK DATA: parameter_nilai
Missing values:
id_parameter_nilai    0
id_level              0
nama_parameter        0
status_parameter      0
dtype: int64

Duplikat id_parameter_nilai: 0 baris

Sample data (5 baris pertama):


,id_parameter_nilai,id_level,nama_parameter,status_parameter
0,P00745,L00022,Class participation,0
1,P00746,L00022,Oral,0
2,P00747,L00022,Listening,0
3,P00748,L00022,Writing,0
4,P00749,L00022,Writing-1,1


✅ Semua id_level di parameter_nilai valid (ditemukan di db_old).


In [20]:
# =================================================
# CEK ANOMALI - parameter_nilai
# =================================================

print("=== CEK idp_nilai ===")
print("Jumlah unik:", df_param_lama['idp_nilai'].nunique())
print("Jumlah total:", len(df_param_lama))
duplikat = df_param_lama[df_param_lama['idp_nilai'].duplicated(keep=False)]
if len(duplikat) > 0:
    print("\n⚠️ Ada duplikat idp_nilai:")
    print(duplikat[['idp_nilai', 'parameter']].head(10))
else:
    print("✅ Tidak ada duplikat idp_nilai.")

# Cek pola: huruf 'P' diikuti 5 digit angka
import re
pola = re.compile(r'^P\d{5}$')
mask_pola = df_param_lama['idp_nilai'].apply(lambda x: bool(pola.match(x)))
if not mask_pola.all():
    print("\n⚠️ idp_nilai tidak sesuai pola P00000:")
    print(df_param_lama[~mask_pola][['idp_nilai']].head())
else:
    print("✅ Semua idp_nilai sesuai format Pxxxxx.")

# Cek spasi ekstra
ada_spasi = df_param_lama['idp_nilai'].str.contains(r'^\s|\s$', regex=True).any()
if ada_spasi:
    print("⚠️ Ada spasi di awal/akhir idp_nilai.")
else:
    print("✅ Tidak ada spasi ekstra di idp_nilai.")

print("\n=== CEK idlevel ===")
print("Jumlah unik idlevel:", df_param_lama['idlevel'].nunique())
pola_level = re.compile(r'^L\d{5}$')
mask_level = df_param_lama['idlevel'].apply(lambda x: bool(pola_level.match(x)))
if not mask_level.all():
    print("\n⚠️ idlevel tidak sesuai pola L00000:")
    print(df_param_lama[~mask_level][['idlevel']].head())
else:
    print("✅ Semua idlevel sesuai format Lxxxxx.")

# Cek apakah idlevel ada di tabel 'level' di db_old?
try:
    level_lama = pd.read_sql('SELECT idlevel FROM level', db_old)
    set_level = set(level_lama['idlevel'].unique())
    missing = df_param_lama[~df_param_lama['idlevel'].isin(set_level)]
    if len(missing) > 0:
        print(f"\n⚠️ {len(missing)} idlevel tidak ditemukan di tabel level:")
        print(missing[['idlevel']].drop_duplicates().head())
    else:
        print("✅ Semua idlevel terdaftar di tabel level (db_old).")
except Exception as e:
    print(f"ℹ️ Tidak bisa validasi ke tabel level: {e}")

print("\n=== CEK isnumber ===")
print("Nilai unik:", df_param_lama['isnumber'].unique())
# harusnya hanya 0.0 dan 1.0

=== CEK idp_nilai ===
Jumlah unik: 1187
Jumlah total: 1187
✅ Tidak ada duplikat idp_nilai.
✅ Semua idp_nilai sesuai format Pxxxxx.
✅ Tidak ada spasi ekstra di idp_nilai.

=== CEK idlevel ===
Jumlah unik idlevel: 142
✅ Semua idlevel sesuai format Lxxxxx.
✅ Semua idlevel terdaftar di tabel level (db_old).

=== CEK isnumber ===
Nilai unik: [0. 1.]


In [21]:
# Mapping & finalisasi (jika sudah bersih)
mapping_param = {
    'idp_nilai': 'id_parameter_nilai',
    'idlevel': 'id_level',
    'parameter': 'nama_parameter',
    'isnumber': 'status_parameter'   # 0.0/1.0 -> integer
}
df_param = df_param_lama.rename(columns=mapping_param)
df_param['status_parameter'] = df_param['status_parameter'].astype(int)
# Tipe lainnya biarkan string karena ID varchar

In [22]:
# Setelah mapping dan konversi status_parameter
# HAPUS kolom id_parameter_nilai karena auto_increment di DB baru
df_param = df_param.drop(columns=['id_parameter_nilai'], errors='ignore')

# Cek hasil akhir
print("Kolom df_param setelah drop:", df_param.columns.tolist())

Kolom df_param setelah drop: ['id_level', 'nama_parameter', 'status_parameter']


In [23]:
print("\n=== RINGKASAN TABEL ===")
print("\n1. Tabel Periode")
print(df_periode.head())
print("\n2. Tabel Parameter")
print(df_param.head())


=== RINGKASAN TABEL ===

1. Tabel Periode
  id_periode                               nama_periode id_kursus  \
0     P00006   General English Term I July-October 2023    K00001   
1     P00008  General English Term II Oct '23 - Feb '24    K00001   
2     P00009      General English Term III Feb-Jun 2024    K00001   
3     P00010                Coding Semester I-2023/2024    K00002   
4     P00011               Coding Semester II-2023/2024    K00002   

   jumlah_sesi tahun_ajar tanggal_mulai  status  is_active  
0           30  2023/2024    2023-07-04       1          1  
1           30  2023/2024    2023-10-25       1          1  
2           30  2023/2024    2024-02-21       1          1  
3           18  2023/2024    2023-08-01       1          1  
4           18  2023/2024    2024-01-23       1          1  

2. Tabel Parameter
  id_level       nama_parameter  status_parameter
0   L00022  Class participation                 0
1   L00022                 Oral                 0
2   L0

In [24]:
# ================================
# 3. GABUNG & EXPORT
# ================================
import pickle

hasil_fase_2 = {
    'periode': df_periode,
    'parameter_nilai': df_param
}

output_path = 'fase_2_afrida.pkl'
with open(output_path, 'wb') as f:
    pickle.dump(hasil_fase_2, f)

print(f"\n💾 Data final fase 2 tersimpan di '{output_path}'")
print("Isi:")
for nama, df in hasil_fase_2.items():
    print(f"   - {nama}: {len(df)} baris, kolom: {list(df.columns)}")


💾 Data final fase 2 tersimpan di 'fase_2_afrida.pkl'
Isi:
   - periode: 91 baris, kolom: ['id_periode', 'nama_periode', 'id_kursus', 'jumlah_sesi', 'tahun_ajar', 'tanggal_mulai', 'status', 'is_active']
   - parameter_nilai: 1187 baris, kolom: ['id_level', 'nama_parameter', 'status_parameter']


In [25]:
with open("fase_2_afrida.pkl", "rb") as f:
    data_loaded = pickle.load(f)

print("📦 Isi file:")
for key in data_loaded.keys():
    print(f" - {key}: {data_loaded[key].shape}")

📦 Isi file:
 - periode: (91, 8)
 - parameter_nilai: (1187, 3)


In [15]:
from IPython.display import display

for key, df in data_loaded.items():
    print(f"\n📊 {key}")
    display(df.head())


📊 periode


,id_periode,nama_periode,id_kursus,jumlah_sesi,tahun_ajar,tanggal_mulai,status,is_active
0,P00006,General English Term I July-October 2023,K00001,30,2023/2024,2023-07-04,1,1
1,P00008,General English Term II Oct '23 - Feb '24,K00001,30,2023/2024,2023-10-25,1,1
2,P00009,General English Term III Feb-Jun 2024,K00001,30,2023/2024,2024-02-21,1,1
3,P00010,Coding Semester I-2023/2024,K00002,18,2023/2024,2023-08-01,1,1
4,P00011,Coding Semester II-2023/2024,K00002,18,2023/2024,2024-01-23,1,1



📊 parameter_nilai


,id_parameter_nilai,id_level,nama_parameter,status_parameter
0,P00745,L00022,Class participation,0
1,P00746,L00022,Oral,0
2,P00747,L00022,Listening,0
3,P00748,L00022,Writing,0
4,P00749,L00022,Writing-1,1


## 3. Transform Data (jika diperlukan)

## 4. Insert ke DB Baru

## 5. Verifikasi Data

## 6. Return Hasil Migrasi untuk migrate_db.py

## 7. Close Connection